# 02 - Baselines B0 y B1

B0 predice la prevalencia estimada en `train`; B1 convierte una puntuacion fija de recursos en probabilidad usando solo `train`. Los escenarios A, B y C se procesan por separado y cada partida tiene el mismo peso total. Este cuaderno no usa `calib` ni `test`.

La formula de B1 fue adoptada para esta rama. El ajuste limita Newton a seis iteraciones y no registra si convergio; esa limitacion acompana sus metricas.

Ejecuta las celdas de arriba abajo. La celda de B0/B1 revisa primero Kaggle: si los resultados ya existen los baja y no recalcula nada.

In [ ]:
# PREPARACION DEL CUADERNO
# PROPOSITO: dejar este servidor de Colab listo SIN depender de otro cuaderno.
# - Clona el repo (o lo actualiza) e instala las librerias fijadas en requirements.txt.
# - Muestra nucleos de CPU y RAM. Para entrenar se espera CPU + RAM alta (8 nucleos, ~51 GiB).
# - Configura Kaggle: token de los secretos de Colab, de secrets.txt o pedido a mano (oculto).
# - Baja a /content/datos solo las tablas que falten en este servidor.
# Los RESULTADOS (CSV, JSON, .joblib) NO se guardan en Drive ni en el repo: viven en el dataset
# privado de Kaggle. Cada celda que entrena revisa SUS archivos en Kaggle: si ya estan, los baja
# a /content/resultados y no entrena; si faltan, entrena y los publica.
# Despues del git pull se "olvidan" los modulos ptcg_wp ya cargados para usar el codigo nuevo.
# ESPERADO: la linea de nucleos/RAM, "ya existe" o "bajado" para las 21 tablas y
# "Tablas listas | caracteristicas: 62".
import json
import os
import subprocess
import sys
import time
from pathlib import Path

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "04MainAIModelTrainingAndDevelopment"
REPO_DIR = Path("/content/repo")
if not REPO_DIR.exists():
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo fetch origin {BRANCH}
    !git -C /content/repo checkout {BRANCH}
    !git -C /content/repo pull --ff-only origin {BRANCH}
%cd /content/repo
!pip install -q -r requirements.txt
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for nombre in [m for m in list(sys.modules) if m == "ptcg_wp" or m.startswith("ptcg_wp.")]:
    del sys.modules[nombre]

# Nucleos y RAM de este servidor, y credenciales de Kaggle
from ptcg_wp.colab import recursos, setup_kaggle
RECURSOS = recursos()
setup_kaggle()

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from ptcg_wp import store
from ptcg_wp.features import feature_columns
from ptcg_wp.ingest import load_days

# Carpetas de trabajo: tablas en /content/datos y resultados en /content/resultados
DATA_DIR = store.DATOS_DIR
RESULTADOS = store.RESULTADOS_DIR
DATASET = store.load_dataset_id()
store.fetch(DATASET, DATA_DIR, store.expected_files(load_days("configs/dias.yaml")))

# Las 62 caracteristicas del modelo principal (sin ids de carta ni columnas meta)
schema = pq.ParquetFile(sorted(DATA_DIR.glob("estados_*.parquet"))[0]).schema.names
FEATURES = feature_columns(schema)
assert len(FEATURES) == 62, len(FEATURES)
SEED = 2026
SCENARIOS = ("A", "B", "C")

# Trazabilidad que se guarda en cada registro.json nuevo
COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
DIRTY = bool(subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip())
RUN_INFO = {"commit": COMMIT, "working_tree_dirty": DIRTY, "seed": SEED,
            "fecha_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "recursos": RECURSOS}
print("Tablas listas | caracteristicas:", len(FEATURES), "| commit:", COMMIT[:7],
      "| cambios locales:", DIRTY)

## Peso igual por partida

Una partida con 200 estados da peso `1/200` a cada estado; una con 20 da `1/20`. Cada partida suma 1 en total. Para una metrica por etapa, los pesos se recalculan dentro de esa etapa.

In [ ]:
# FUNCION DE METRICAS
# PROPOSITO: evaluar globales y por etapa con el modulo comun de todos los baselines.
from ptcg_wp.evaluation import metric_rows


def evaluar(val, pred, escenario, modelo):
    return metric_rows(val["y"].to_numpy(), np.asarray(pred), val["game_id"].to_numpy(),
                       val["stage"].to_numpy(), {"escenario": escenario, "modelo": modelo})


print("Funciones comunes de metricas listas.")

## B0 - probabilidad a priori

B0 ignora el tablero. Calcula en train la fraccion de partidas ganadas por quien actua y predice esa probabilidad en todos los estados de val. Es el piso de comparacion.

## B1 - heuristica de recursos

**Formula adoptada:** `score = (opp_prizes - own_prizes) + diff_total_hp / hp_unit + diff_bench_n`. El signo queda positivo cuando el estado favorece al jugador que actua. `hp_unit` es el promedio de `active_max_hp` positivo observado en `train`, ponderado para que cada partida pese igual. Se calcula por escenario y no usa `y` ni los ids de carta.

Despues, una regresion logistica de una sola variable aprende en train la pendiente y el intercepto que convierten ese score en probabilidad. No aprende pesos separados para premios, PV y banca.

**Por que no hay `.joblib`:** B0 es un numero y B1 son tres numeros mas la formula. `ajustes_B0_B1.csv` los define por completo y `ptcg_wp.baselines.b1_probability` reproduce las predicciones.

In [ ]:
# B0 Y B1: AJUSTE EN TRAIN Y MEDICION EN VAL, POR ESCENARIO
# PROPOSITO: obtener metricas y parametros de B0/B1.
# - Si los 3 archivos de b0_b1/ ya estan en Kaggle: se bajan y la celda termina SIN calcular.
# - Si falta alguno: se calculan B0/B1 para A, B y C (~2 min) y se publican en Kaggle.
# ESPERADO (val): B0 AUC 0.5 en todos; B1 AUC 0.6768 (A), 0.6677 (B), 0.6714 (C).
import gc
from ptcg_wp.baselines import ScenarioStream, b0_and_hp_unit, b1_probability, fit_b1_mapping

SALIDA_B0_B1 = RESULTADOS / "b0_b1"
ESPERADOS_B0_B1 = ["b0_b1/metricas_B0_B1.csv", "b0_b1/ajustes_B0_B1.csv", "b0_b1/registro.json"]


def entrenar_b0_b1():
    SALIDA_B0_B1.mkdir(parents=True, exist_ok=True)
    resultados, ajustes, tiempos = [], [], {}
    columnas = ["game_id", "y", "own_prizes", "opp_prizes", "own_active_max_hp",
                "opp_active_max_hp", "diff_total_hp", "diff_bench_n"]  # la etapa se deriva del turno
    for escenario in SCENARIOS:
        inicio = time.perf_counter()
        stream = ScenarioStream(DATA_DIR, escenario, tuple(columnas))  # un dia a la vez: poca RAM
        prevalencia, hp_unit = b0_and_hp_unit(stream)
        beta_b1 = fit_b1_mapping(stream, hp_unit, prevalencia)
        partes_val = []
        for dia in stream:
            mascara_val = dia["split"] == "val"
            if mascara_val.any():
                val = dia.loc[mascara_val, ["game_id", "y", "stage"]].copy()
                val["pred_b1"] = b1_probability(dia.loc[mascara_val], hp_unit, beta_b1)
                partes_val.append(val)
        val = pd.concat(partes_val, ignore_index=True)
        resultados += evaluar(val, np.full(len(val), prevalencia), escenario, "B0")
        resultados += evaluar(val, val["pred_b1"].to_numpy(), escenario, "B1")
        ajustes.append({"escenario": escenario, "prevalencia_B0_train": prevalencia,
                        "hp_unit_train": hp_unit, "intercepto_B1": beta_b1[0],
                        "pendiente_B1": beta_b1[1], "partidas_val": val.game_id.nunique()})
        tiempos[escenario] = time.perf_counter() - inicio
        print(f"Escenario {escenario}: listo en {tiempos[escenario]:.0f} s")
        del stream, partes_val, val
        gc.collect()
    pd.DataFrame(resultados).to_csv(SALIDA_B0_B1 / "metricas_B0_B1.csv", index=False)
    pd.DataFrame(ajustes).to_csv(SALIDA_B0_B1 / "ajustes_B0_B1.csv", index=False)
    registro = {**RUN_INFO, "escenarios": list(SCENARIOS), "split_fit": "train", "split_eval": "val",
                "ponderacion": "cada partida suma 1 en cada subconjunto",
                "B1_formula": "(opp_prizes - own_prizes) + diff_total_hp / hp_unit + diff_bench_n",
                "B1_estado": "formula_adoptada_rama_04", "segundos_por_escenario": tiempos}
    (SALIDA_B0_B1 / "registro.json").write_text(json.dumps(registro, indent=2, ensure_ascii=False))


store.asegurar(ESPERADOS_B0_B1, entrenar_b0_b1, "B0/B1: metricas y ajustes")
display(pd.read_csv(SALIDA_B0_B1 / "metricas_B0_B1.csv").round(4))
display(pd.read_csv(SALIDA_B0_B1 / "ajustes_B0_B1.csv").round(4))